# PanAf Ape Detection — Phase 1 "See"

**Pretrained MegaDetector V6 over all 10 PanAf500 clips, on a Colab GPU.**

## How to run this

1. **Runtime → Change runtime type → T4 GPU → Save.** Do this *first* — changing it later restarts
   the session and throws away everything installed.
2. **Runtime → Run all.**
3. Wait ~15 minutes. If Colab asks you to restart the session after the install, click **Restart**,
   then **Runtime → Run all** again — completed steps are skipped, so it picks up where it left off.

Nothing needs editing. No files to upload.

### What it does

Downloads 10 PanAf500 clips (~23 MB) straight from the Bristol deposit, runs MegaDetector over
**every frame of all 10** (3,600 frames), stitches annotated video, and measures accuracy against
the dataset's ground-truth boxes.

### Two things to know

- **The device.** PyTorch-Wildlife accepts `device="cuda"`, stores it, and **never applies it** —
  the weights load on CPU and nothing raises. This notebook forces and *verifies* the placement, so
  watch for `weights forced onto cuda:0` in section 5.
- **Green vs amber.** In the annotated video, **green = MegaDetector prediction**,
  **amber = dataset ground truth** with its behaviour label. MegaDetector only ever outputs
  `animal` — not species, not behaviour.

### Licence

PanAf20K is under a **Non-Commercial Government Licence v2**. Clips downloaded here must not be
redistributed, and the annotated video is a derived work.

## 1. Check the GPU

In [ ]:
import subprocess

out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode == 0:
    print([l for l in out.stdout.splitlines() if "MiB" in l or "Tesla" in l or "NVIDIA" in l][:2])
    print("\nGPU OK.")
else:
    raise SystemExit(
        "NO GPU. Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all."
    )

## 2. Get the code

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/adikothuri3/PanAF-Ape-Detection.git"
REPO_DIR = "/content/PanAF-Ape-Detection"

if not Path(REPO_DIR).exists():
    !git clone --depth 1 -q $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull -q

os.chdir(REPO_DIR)
os.environ["PANAF_REPO_ROOT"] = REPO_DIR
print("working in", Path.cwd())

## 3. Install

**Deliberately does not install `requirements-colab.txt` here.** That file pins the full 170-package
locked environment including `torch`, and forcing it onto Colab replaces the CUDA-matched torch that
is already installed — a multi-gigabyte download that can leave the runtime without working CUDA.

Instead this installs only what Colab lacks and keeps Colab's torch. The locked file remains the
source of truth for reproducing the environment *outside* Colab.

Two or three minutes.

In [ ]:
# Keep Colab's CUDA-matched torch; add only what is missing.
# setuptools<81 because yolov5 (via PytorchWildlife) still imports pkg_resources,
# which setuptools 81 deprecated and 83 removed.
!pip install -q "setuptools<81" 2>&1 | tail -2
!pip install -q pytorchwildlife soundfile librosa 2>&1 | tail -2
!pip install -q -e . --no-deps 2>&1 | tail -2
print("install finished")

In [ ]:
# Confirm torch still sees the GPU after the install.
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("\nCUDA is missing. Runtime > Restart session, then Runtime > Run all.")

## 4. Verify the stack

`smoke_inference.py` proves the heavy stack actually works — imports, ByteTrack, NumPy interop, a
video round-trip — **without downloading any model weights**. If something is wrong with the
environment, it fails here in seconds rather than twenty minutes into the run.

In [ ]:
!python scripts/smoke_inference.py

## 5. Download the clips

Straight from the Bristol deposit — no upload needed. ~23 MB.

Selection is **purposive, not random**: it profiles candidate annotations first (no video), then
greedily picks clips covering all nine behaviours, both species, crowded frames, small and large
subjects, and frames containing no ape. The reason for each pick goes into the manifest.

In [ ]:
!python scripts/fetch_panaf500.py --count 10 --pool 150

In [ ]:
import pandas as pd

manifest = pd.read_csv("data/sample_manifest.csv")
print(f"{len(manifest)} clips selected\n")
for _, row in manifest.iterrows():
    print(f"{row.clip_id}  [{row.split}]  {row.species}")
    print(f"    {row.selected_reason}\n")

## 6. Run detection on all 10 clips

Every frame of every clip: decode → MegaDetector → confidence filter → compare against ground truth
→ draw boxes → stitch to MP4 → write metrics and run metadata.

**~10–15 minutes.** Watch for the device line early on:

```
WARNING ... PyTorch-Wildlife ignored device='cuda' (weights on 'cpu'); forcing it
INFO    ... weights forced onto cuda:0
```

That is the upstream bug being corrected. If it instead says the weights stayed on CPU, stop — the
run would be ~20x slower and the metadata would be wrong.

Clips already finished are skipped, so re-running after a dropped session resumes.

In [ ]:
!panaf-phase1 detect --config configs/colab.yaml

## 7. Results

Real measurements at the stated confidence and IoU thresholds. A detection counts as correct when it
**localises** an annotated ape — MegaDetector cannot identify species, so no species claim is made.

In [ ]:
import json
from pathlib import Path

import pandas as pd

metrics = [json.loads(p.read_text()) for p in sorted(Path("artifacts/metrics").glob("*.json"))]

table = pd.DataFrame([{
    "clip": m["clip_id"],
    "frames": m["frames_evaluated"],
    "precision": round(m["overall"]["precision"], 3),
    "recall": round(m["overall"]["recall"], 3),
    "f1": round(m["overall"]["f1"], 3),
    "mean_iou": round(m["mean_iou"], 3),
    "empty_frames": m["empty_frames"],
    "FP_on_empty": m["false_positives_on_empty_frames"],
} for m in metrics])
display(table)

tp = sum(m["overall"]["true_positives"] for m in metrics)
fp = sum(m["overall"]["false_positives"] for m in metrics)
fn = sum(m["overall"]["false_negatives"] for m in metrics)
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

print(f"\n{len(metrics)} clips, {sum(m['frames_evaluated'] for m in metrics)} frames")
print(f"TP={tp}  FP={fp}  FN={fn}")
print(f"precision={precision:.3f}  recall={recall:.3f}  F1={f1:.3f}")

### Where it fails

This is the table that should drive any fine-tuning decision. A detector that misses arboreal
postures and small subjects needs different work from one that misses everything equally.

In [ ]:
from collections import defaultdict

behaviour = defaultdict(lambda: [0, 0])
size = defaultdict(lambda: [0, 0])

for m in metrics:
    for label, c in m["by_behaviour"].items():
        behaviour[label][0] += c["true_positives"]
        behaviour[label][1] += c["true_positives"] + c["false_negatives"]
    for band, c in m["by_size"].items():
        size[band][0] += c["true_positives"]
        size[band][1] += c["true_positives"] + c["false_negatives"]

print("Recall by behaviour (worst first)")
for label, (found, total) in sorted(behaviour.items(), key=lambda kv: kv[1][0] / max(kv[1][1], 1)):
    print(f"  {label:20} {found:5}/{total:<6} {found / total:.3f}")

print("\nRecall by subject size (fraction of frame area)")
for band in ("small", "medium", "large"):
    if band in size:
        found, total = size[band]
        print(f"  {band:8} {found:5}/{total:<6} {found / total:.3f}")

## 8. Watch an annotated clip

**Green = MegaDetector prediction** (with confidence). **Amber = dataset ground truth** (with the
behaviour label and the individual's id). The legend is drawn on every frame, so a still pulled out
of the video is still unambiguous about which box came from where.

In [ ]:
import base64
from pathlib import Path

from IPython.display import HTML, display

videos = sorted(Path("artifacts/videos").glob("*_annotated.mp4"))
print(f"{len(videos)} annotated clips in artifacts/videos/\n")

# Colab's player cannot decode mp4v, so re-encode to H.264 just for display.
for source in videos[:2]:
    playable = source.with_name(source.stem + "_h264.mp4")
    !ffmpeg -y -loglevel error -i "{source}" -vcodec libx264 -pix_fmt yuv420p "{playable}"
    encoded = base64.b64encode(playable.read_bytes()).decode()
    print(source.name)
    display(HTML(
        f'<video width=720 controls><source src="data:video/mp4;base64,{encoded}" '
        f'type="video/mp4"></video>'
    ))

## 9. Keep the outputs

**Colab sessions are ephemeral — anything not copied out is lost.**

Set `USE_DRIVE = True` and re-run this cell to copy `artifacts/` to your Drive. You will be asked to
authorise access.

Do not commit the clips or the annotated video: they are derived works of a non-commercially
licensed dataset, and `artifacts/` and `data/` are git-ignored for that reason.

In [ ]:
USE_DRIVE = False
DESTINATION = "/content/drive/MyDrive/panaf-ape-detection/artifacts"

if USE_DRIVE:
    import shutil

    from google.colab import drive

    drive.mount("/content/drive")
    shutil.copytree("artifacts", DESTINATION, dirs_exist_ok=True)
    print("copied to", DESTINATION)
else:
    print("Drive copy off. Set USE_DRIVE = True and re-run this cell to keep the outputs.")

In [ ]:
# The run-metadata record: commit, config, verified device, variant, threshold,
# seed, input checksums, elapsed time. This is what makes the run reproducible.
import json
from pathlib import Path

for path in sorted(Path("artifacts/metadata").glob("*.json"))[-1:]:
    meta = json.loads(path.read_text())
    for key in ("experiment_name", "git_commit", "git_dirty", "device", "model_variant",
                "confidence_threshold", "seed", "elapsed_seconds"):
        print(f"{key:22} {meta.get(key)}")
    print(f"{'inputs':22} {len(meta.get('inputs', []))} files, checksummed")

## Next

- Record what you saw in `experiments/experiment_log.md`, **including anything that failed**.
- Compare these numbers against the local MPS run in
  `reports/phase1_findings_2026-07-25.md`. They should agree closely; a large gap would mean the
  device or the pipeline differs between the two.
- **Before any fine-tuning**, run the two cheap experiments: a confidence-threshold sweep
  (`panaf-phase1 evaluate` recomputes from saved detections, so it costs no GPU time) and a variant
  comparison (`MDV6-yolov10-e`, a config-only change).

Nothing here says anything about species. If the recall table shows a specific weakness, that is the
evidence a fine-tuning decision should rest on.